# CEOAI:  - Baseline
## "Project KRAKEN" Implementation

### Task Summary
We are processing sensor data from a singularity (X-99). The goal is to solve three subtasks:
1.  **Geodesic Rectification**: Predict 10 spline coefficients to un-warp time.
2.  **Topological Classification**: Identify the entity class (0-49) or "Abyssal" (-1).
3.  **Heisenberg Stability**: Predict a stability scalar (0.0 to 1.0).

### Dataset
- **Slices**: (N, 3, 128, 128) - 2D holographic data.
- **Echoes**: (N, 1024, 2) - Gravitational wave time series.
- **Glyphs**: Metadata/Telemetry (CSV).
- **Targets**: Ground truth labels.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Configuration
DATA_DIR = "./data"
SUBMISSION_FILE = "submission.csv"

# Ensure data directory exists
os.makedirs(DATA_DIR, exist_ok=True)

print("Environment setup complete.")

Environment setup complete.


In [ ]:
print("Loading datasets...")

# Load Binary Files (Numpy)
X_train_slices = np.load(f"{DATA_DIR}/train_slices.npy")
X_test_slices = np.load(f"{DATA_DIR}/test_slices.npy")

X_train_echoes = np.load(f"{DATA_DIR}/train_echoes.npy")
X_test_echoes = np.load(f"{DATA_DIR}/test_echoes.npy")

# Load Tables
df_train_targets = pd.read_csv(f"{DATA_DIR}/train_targets.csv")
df_test_glyphs = pd.read_csv(f"{DATA_DIR}/test_glyphs.csv")

# Parse the Training Targets
# Subtask 1 targets need to be converted from string "0.1;0.2..." to numpy arrays
def parse_coeffs(row):
    return np.array([float(x) for x in row.split(';')])

# Apply parsing
y_train_s1 = np.stack(df_train_targets['s1_coeffs'].apply(parse_coeffs).values)
y_train_s2 = df_train_targets['s2_class'].values
y_train_s3 = df_train_targets['s3_stability'].values

print(f"Train Slices Shape: {X_train_slices.shape}")
print(f"Train Echoes Shape: {X_train_echoes.shape}")
print(f"Targets Loaded: {len(df_train_targets)}")

In [ ]:
# Visualize a single datapoint (Index 0)
idx = 0

fig, ax = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f"Datapoint {idx} - Raw Sensor Data", fontsize=16)

# 1. Visualizing Slices (The 2D Holograms)
# Channel 0: Gravimetric Potential (Real)
ax[0, 0].imshow(X_train_slices[idx, 0, :, :], cmap='magma')
ax[0, 0].set_title("Channel 0: Gravimetric (Real)")

# Channel 1: Frame-dragging (Imaginary)
ax[0, 1].imshow(X_train_slices[idx, 1, :, :], cmap='twilight')
ax[0, 1].set_title("Channel 1: Frame-Dragging (Imag)")

# 2. Visualizing Echoes (Gravitational Waves)
# Plotting the Plus (+) and Cross (x) polarization over time
time_steps = np.arange(1024)
ax[1, 0].plot(time_steps, X_train_echoes[idx, :, 0], label='h+ (Plus)', alpha=0.7)
ax[1, 0].plot(time_steps, X_train_echoes[idx, :, 1], label='hx (Cross)', alpha=0.7, color='orange')
ax[1, 0].set_title("Gravitational Wave Echoes")
ax[1, 0].legend()
ax[1, 0].set_xlabel("Time (t)")

# 3. Distribution of Classes (Target)
unique, counts = np.unique(y_train_s2, return_counts=True)
ax[1, 1].bar(unique, counts)
ax[1, 1].set_title("Class Distribution (Subtask 2)")
ax[1, 1].set_xlabel("Class ID (-1 = Abyssal)")

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# NAIVE BASELINE MODEL
# Strategy: Ignore input features. Predict Mean/Mode of training set.
# ---------------------------------------------------------------

print("Training Naive Models...")

# Subtask 1: Geodesic Rectification (Regression)
# Strategy: Calculate the average coefficient vector across all training examples.
# If the warping is somewhat consistent, this gives a baseline score.
mean_coeffs = np.mean(y_train_s1, axis=0)
print(f"Subtask 1 Baseline: Predicting constant vector {mean_coeffs[:2]}...")

# Subtask 2: Topological Classification (Classification)
# Strategy: Predict the Mode (Most Frequent Class).
# In a real scenario, "Abyssal" (-1) might be rare or common.
from scipy import stats
mode_class = int(stats.mode(y_train_s2)[0])
print(f"Subtask 2 Baseline: Predicting majority class {mode_class}")

# Subtask 3: Heisenberg Stability (Regression)
# Strategy: Predict the global mean stability.
mean_stability = float(np.mean(y_train_s3))
print(f"Subtask 3 Baseline: Predicting mean stability {mean_stability:.4f}")

In [ ]:
# ---------------------------------------------------------------
# GENERATE SUBMISSION FILE
# ---------------------------------------------------------------

submission_rows = []

# Get list of test IDs from the glyphs file (or just generate based on N_TEST)
test_ids = df_test_glyphs['datapointID'].values

print(f"Generating predictions for {len(test_ids)} test items...")

for tid in test_ids:
    # --- Subtask 1: Coefficients ---
    # Format: "f1;f2;f3..."
    # We use our calculated mean_coeffs
    s1_pred_str = ";".join([f"{x:.6f}" for x in mean_coeffs])
    submission_rows.append([1, tid, s1_pred_str])

    # --- Subtask 2: Class ---
    # We use our calculated mode_class
    submission_rows.append([2, tid, mode_class])

    # --- Subtask 3: Stability ---
    # We use our calculated mean_stability
    submission_rows.append([3, tid, f"{mean_stability:.6f}"])

# Create DataFrame
df_submission = pd.DataFrame(submission_rows, columns=['subtaskID', 'datapointID', 'answer'])

# Save
df_submission.to_csv(SUBMISSION_FILE, index=False)

print("------------------------------------------------")
print(f"Submission saved to {SUBMISSION_FILE}")
print("Head of submission:")
print(df_submission.head())